# AMOC conceptual model

AMOC panels using the bundled generator `generate()` with the bundled AMOC box model. **Deterministic mode** (`sigma_tilde=0`) so tip vs non-tip is reliable. Layout: **rate-induced** = 3 panels (Q(t), H(t), phase S_N vs S_T); **bifurcation-induced** = 2 panels (Q(t), H(t)).

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "examples"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from experiments.indicators import rolling_variance, rolling_acf, spectral_gap_series


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from experiments.systems import generate
import kews
from kews.observables import rff, time_delay, mlp

LABEL_FONTSIZE = 16
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13


In [ ]:
# Global parameters
n_samples = 50
num_timesteps = 500
burnin_idx = 0
sigma_tilde = 1.0

def extract_1d(traj):
    """Extract 1D state from trajectory (may be 2D: [time, state_dim])."""
    if traj.ndim > 1:
        return traj[:, 0]
    return traj


# 1. Rate-induced Tipping

## Generate Trajectories

Generate tipping and non-tipping trajectories using the AMOC box model (rate-induced)

In [ ]:
# Generate tipping trajectories (state + control lambda(t) for each)
# Full: for EWS (window of num_timesteps ending at cutoff_idx) and for visualization.
tipping_trajectories_viz = []
tipping_full_params = []
tipping_indices = []
tipping_states = []
key = jax.random.PRNGKey(42)

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, tip_info = generate(
        subkey,
        tipping_type='r_amoc',
        null_simulation=False,
        sigma_tilde=sigma_tilde,
        num_timesteps=num_timesteps
    )
    tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))
    tipping_indices.append(tip_info.get('transition_index', None))
    tipping_states.append(tip_info.get('state_trajectory'))

# Generate non-tipping trajectories (state + control for each)
non_tipping_trajectories_viz = []
non_tipping_full_params = []
non_tipping_states = []

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, nontip_info = generate(
        subkey,
        tipping_type='r_amoc',
        null_simulation=True,
        sigma_tilde=sigma_tilde,
        num_timesteps=num_timesteps
    )
    non_tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    non_tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))
    non_tipping_states.append(nontip_info.get('state_trajectory'))

# Universal cutoff (earliest transition index)
cutoff_idx = min(idx - burnin_idx for idx in tipping_indices if idx is not None)
start_ews = max(0, cutoff_idx - num_timesteps)

# EWS window: for each trajectory take time slice [start_ews:cutoff_idx]
tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in tipping_trajectories_viz])
tipping_controls = np.array([p[start_ews:cutoff_idx] for p in tipping_full_params])
# tipping_controls = (tipping_controls - tipping_controls.mean(axis=1, keepdims=True)) / (tipping_controls.std(axis=1, keepdims=True) + 1e-12)

non_tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in non_tipping_trajectories_viz])
non_tipping_controls = np.array([p[start_ews:cutoff_idx] for p in non_tipping_full_params])
# non_tipping_controls = (non_tipping_controls - non_tipping_controls.mean(axis=1, keepdims=True)) / (non_tipping_controls.std(axis=1, keepdims=True) + 1e-12)


## Visualize Trajectories

Compare tipping vs non-tipping trajectories.

In [ ]:
# Plot tipping trajectories (full trajectory, visualization only)
tip_arr = np.asarray(tipping_trajectories_viz)
nontip_arr = np.asarray(non_tipping_trajectories_viz)
x = np.arange(tip_arr.shape[1])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

# 1) Phase space
for i, (tip_states, nontip_states) in enumerate(zip(tipping_states, non_tipping_states)):
    ax[0].plot(
        tip_states[..., 0],
        tip_states[..., 1],
        color="C3",
        alpha=0.5,
        label="Tipping" if i == 0 else None,
    )
    ax[0].plot(
        nontip_states[..., 0],
        nontip_states[..., 1],
        color="C0",
        alpha=0.5,
        label="Non-tipping" if i == 0 else None,
    )
    ax[0].plot(
        nontip_states[0, 0],
        nontip_states[0, 1],
        marker="*",
        markersize=10,
        color="lightgrey",
        linestyle="None",
        label="Initial condition" if i == 0 else None,
    )

ax[0].set_xlabel("North Atlantic salinity [psu]", fontsize=LABEL_FONTSIZE)
ax[0].set_ylabel("Tropical Atlantic salinity [psu]", fontsize=LABEL_FONTSIZE)
ax[0].grid(alpha=0.2)
ax[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

# 2) Time series: envelope + mean
ax[1].fill_between(
    x,
    tip_arr.min(axis=0),
    tip_arr.max(axis=0),
    color="C3",
    alpha=0.5,
    linewidth=0,
    label="Tipping",
)
ax[1].plot(x, tip_arr.mean(axis=0), color="C3", lw=3)

ax[1].fill_between(
    x,
    nontip_arr.min(axis=0),
    nontip_arr.max(axis=0),
    color="C0",
    alpha=0.5,
    linewidth=0,
    label="Non-tipping",
)
ax[1].plot(x, nontip_arr.mean(axis=0), color="C0", lw=3)

ax[1].axvline(cutoff_idx, color="0.4", linestyle="--", lw=2, label="Tipping point")
ax[1].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[1].set_ylabel("AMOC Strength [Sv]", fontsize=LABEL_FONTSIZE)
ax[1].grid(alpha=0.2)
ax[1].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

# 3) Parameter
ax[2].plot(tipping_full_params[0], color="C3", lw=3, label=r"Tipping $H(t)$")
ax[2].plot(non_tipping_full_params[0], color="C0", lw=3, label=r"Non-tipping $H(t)$")
ax[2].axvline(cutoff_idx, color="0.4", linestyle="--", lw=2)
ax[2].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[2].set_ylabel("Parameter", fontsize=LABEL_FONTSIZE)
ax[2].grid(alpha=0.2)
ax[2].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

for a in ax:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
    a.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/r_amoc3b_traj.png", dpi=300, bbox_inches="tight")
plt.show()


## Compute Early Warning Signals

Apply EWS indicators (AR1, Variance) to each trajectory and measure trend with Kendall τ.

In [ ]:
# EWS parameters
wL = 0.5  # Rolling window length (fraction of time series)
N_ENSEMBLES = 1 # Number of ensembles for the ResKMD runs



def compute_ews_metrics(
    trajectories,
    wL,
    controls=None,
    control_method="augmented",
    augmented_future_input="future",
):
    ar1_series = []
    var_series = []
    resdmd_series = []
    spectral_gap_list = []
    resdmd_control_series = [] if controls is not None else None
    resdmd_control_mlp_series = [] if controls is not None else None

    for i, traj in enumerate(trajectories):
        data = np.asarray(traj)
        
        ar1 = rolling_acf(data, window=wL)
        var = rolling_variance(data, window=wL)
        spectral_gap = spectral_gap_series(data, wL)
        
        resdmd_ens = []
        for seed in range(N_ENSEMBLES):
            resdmd = kews.resdmd(data, 
                window=wL,
                observable=time_delay(n_delay=20),
                
            )
            resdmd_ens.append(np.array(resdmd))
    
        ar1_series.append(np.array(ar1))
        var_series.append(np.array(var))
        resdmd_series.append(np.array(resdmd_ens).mean(axis=0))
        spectral_gap_list.append(spectral_gap)
    
        if controls is not None:
            u = np.asarray(controls[i])
            
            resdmd_c = kews.resdmd(data, 
                window=wL,
                control=u,
                observable=time_delay(n_delay=20),
                control_method=control_method,
                future_input=augmented_future_input,
                
            )
    
            resdmd_control_mlp_ens = []
            for seed in range(N_ENSEMBLES):
                resdmd_c_mlp = kews.resdmd(data, 
                    window=wL,
                    control=u,
                    observable=mlp(hidden_layers=[8, 4], bottleneck_dim=2, seed=seed),
                    fit_kwargs=dict(
                        lr=1e-3,
                        epochs=1000,
                        batch_size=32,
                        shuffle=True,
                    ),
                    control_method=control_method,
                    future_input=augmented_future_input,
                )
                resdmd_control_mlp_ens.append(np.array(resdmd_c_mlp))
    
            resdmd_control_series.append(np.array(resdmd_c))
            resdmd_control_mlp_series.append(np.array(resdmd_control_mlp_ens).mean(axis=0))
    
    out = {
        "ar1_series": ar1_series,
        "var_series": var_series,
        "resdmd_series": resdmd_series,
        "spectral_gap_series": spectral_gap_list,
    }
    if resdmd_control_series is not None:
        out["resdmd_control_series"] = resdmd_control_series
        out["resdmd_control_mlp_series"] = resdmd_control_mlp_series
    return out


tipping_ews = compute_ews_metrics(
    tipping_trajectories,
    wL,
    controls=tipping_controls,
    control_method="augmented",
    augmented_future_input="future",
)

non_tipping_ews = compute_ews_metrics(
    non_tipping_trajectories,
    wL,
    controls=non_tipping_controls,
    control_method="augmented",
    augmented_future_input="future",
)


## Visualize EWS Indicators

Compare AR1 and Variance trends between tipping and non-tipping trajectories.

In [ ]:
# Plot EWS
start_idx = 0
t = burnin_idx + start_ews + np.arange(len(tipping_ews['var_series'][0]))

def masked_array(series_list, start_idx):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    return arr

def plot_distribution(ax, series_list, color, label, start_idx, band_alpha=0.15):
    arr = masked_array(series_list, start_idx)
    median = np.nanmedian(arr, axis=0)
    q25, q75 = np.nanpercentile(arr, [25, 75], axis=0)
    ax.plot(t, median, color=color, label=label, linewidth=3)
    ax.fill_between(t, q25, q75, color=color, alpha=band_alpha, linewidth=0)

def robust_ylim(series_lists, start_idx, low=2, high=98, margin=0.05):
    """Compute ylim from percentile of valid values so end spikes don't dominate."""
    vals = np.concatenate([masked_array(s, start_idx)[:, start_idx:].ravel() for s in series_lists])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad

panel_specs = [
    ('spectral_gap_series', 'Spectral gap', False),
    ('var_series', 'Variance', True),
    ('resdmd_series', 'ResKMD (TDF)', True),
    ('resdmd_control_series', 'ResKMD with control (TDF)', True),
    ('resdmd_control_mlp_series', 'ResKMD with control (MLP)', True),
]

fig, axes = plt.subplots(1, len(panel_specs), figsize=(24, 4), sharex=True)

for i, (key, ylabel, use_robust_ylim) in enumerate(panel_specs):
    ax = axes[i]

    plot_distribution(ax, tipping_ews[key], "C3", "Tipping", start_idx)
    plot_distribution(ax, non_tipping_ews[key], "C0", "Non-tipping", start_idx)

    ax.set_xlabel("Time", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_xlim([t[0], t[-1]])
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

    if use_robust_ylim:
        ax.set_ylim(robust_ylim([tipping_ews[key], non_tipping_ews[key]], start_idx))

    if i == 0:
        ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_amoc3b_ews.png', dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# AUROC benchmark across trajectories (tail mean over last k rolling windows)
import pandas as pd
from experiments.metrics import ensemble_roc, plot_auroc, plot_roc

TAIL_K = 3

def tail_mean_finite(series_list, k=TAIL_K):
    out = []
    for series in series_list:
        arr = np.asarray(series, dtype=float)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            out.append(np.nan)
        else:
            out.append(finite[-min(k, finite.size):].mean())
    return np.asarray(out, dtype=float)

y_true = np.r_[np.ones(len(tipping_ews['var_series'])), np.zeros(len(non_tipping_ews['var_series']))].astype(int)

result = ensemble_roc(
    y_true,
    {
        'Variance': np.r_[tail_mean_finite(tipping_ews['var_series']), tail_mean_finite(non_tipping_ews['var_series'])],
        'ResKMD (TDF)': np.r_[tail_mean_finite(tipping_ews['resdmd_series']), tail_mean_finite(non_tipping_ews['resdmd_series'])],
        'ResKMDc (TDF)': np.r_[tail_mean_finite(tipping_ews['resdmd_control_series']), tail_mean_finite(non_tipping_ews['resdmd_control_series'])],
        'ResKMDc (MLP)': np.r_[tail_mean_finite(tipping_ews['resdmd_control_mlp_series']), tail_mean_finite(non_tipping_ews['resdmd_control_mlp_series'])],
    },
)

auroc_df = pd.DataFrame(
    [
        {
            'indicator': name,
            'auroc': curve.auroc,
            'n_valid': curve.n_valid,
            'n_missing': curve.n_missing,
        }
        for name, curve in result.curves.items()
    ]
).sort_values('auroc', ascending=False).reset_index(drop=True)
display(auroc_df)

fig, ax = plt.subplots(figsize=(6, 4))
plot_auroc(result, ax=ax, label_fontsize=LABEL_FONTSIZE, tick_labelsize=TICK_FONTSIZE)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_amoc3b_auroc.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(
    result,
    ax=ax,
    title=None,
    chance=True,
    show_auroc=True,
    linewidth=3,
    bold_top_n=2,
    label_fontsize=LABEL_FONTSIZE,
    tick_labelsize=TICK_FONTSIZE,
    legend_fontsize=LEGEND_FONTSIZE,
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_amoc3b_roc.png', dpi=300, bbox_inches='tight')
plt.show()


# 2. Bifurcation-induced Tipping


## Generate Trajectories

Generate tipping and non-tipping trajectories using the AMOC box model (bifurcation-induced)

In [ ]:
# Generate tipping trajectories (bifurcation-induced: no control used for EWS)
tipping_trajectories_viz = []
tipping_full_params = []  # for optional viz of h(t)
tipping_indices = []
tipping_states_bifurc = []
non_tipping_states_bifurc = []

key = jax.random.PRNGKey(42)

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, info = generate(
        subkey,
        tipping_type='b_amoc',
        sigma_tilde=sigma_tilde,
        null_simulation=False,
        num_timesteps=num_timesteps
    )
    tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))
    tipping_indices.append(info.get('transition_index', None))
    tipping_states_bifurc.append(np.array(full_result.trajectory[burnin_idx:], dtype=np.float64))

# Generate non-tipping (null) trajectories
non_tipping_trajectories_viz = []
non_tipping_full_params = []

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result, part_result, info = generate(
        subkey,
        tipping_type='b_amoc',
        sigma_tilde=sigma_tilde,
        null_simulation=True,
        num_timesteps=num_timesteps
    )
    non_tipping_trajectories_viz.append(extract_1d(full_result.trajectory[burnin_idx:]))
    non_tipping_full_params.append(np.array(full_result.param_values[burnin_idx:], dtype=np.float64))
    non_tipping_states_bifurc.append(np.array(full_result.trajectory[burnin_idx:], dtype=np.float64))

# Universal cutoff (earliest transition index) and EWS window
cutoff_idx = min(idx - burnin_idx for idx in tipping_indices if idx is not None)
start_ews = max(0, cutoff_idx - num_timesteps)

# EWS window: [start_ews:cutoff_idx] (same as saddle / rate-induced)
tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in tipping_trajectories_viz])
non_tipping_trajectories = np.array([t[start_ews:cutoff_idx] for t in non_tipping_trajectories_viz])

tipping_states_bifurc = np.array(tipping_states_bifurc)
non_tipping_states_bifurc = np.array(non_tipping_states_bifurc)


## Visualize Trajectories

Compare tipping vs non-tipping trajectories.

In [ ]:
tip_arr = np.asarray(tipping_trajectories_viz)
nontip_arr = np.asarray(non_tipping_trajectories_viz)
x = np.arange(tip_arr.shape[1])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

# 1) Phase space
for i, (tip_states, nontip_states) in enumerate(
    zip(tipping_states_bifurc, non_tipping_states_bifurc)
):
    # tip_states, nontip_states have shape (T, 3): [Q, S_N, S_T]
    ax[0].plot(
        tip_states[:, 1],
        tip_states[:, 2],
        alpha=0.5,
        color="C3",
        label="Tipping" if i == 0 else None,
    )
    ax[0].plot(
        nontip_states[:, 1],
        nontip_states[:, 2],
        alpha=0.5,
        color="C0",
        label="Non-tipping" if i == 0 else None,
    )
    if i == 0:
        ax[0].plot(
            nontip_states[0, 1],
            nontip_states[0, 2],
            marker="*",
            markersize=10,
            color="lightgrey",
            linestyle="None",
            label="Initial condition",
        )

ax[0].set_xlabel("North Atlantic salinity [psu]", fontsize=LABEL_FONTSIZE)
ax[0].set_ylabel("Tropical Atlantic salinity [psu]", fontsize=LABEL_FONTSIZE)
ax[0].grid(alpha=0.2)
ax[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

# 2) State time series: envelope + mean
ax[1].fill_between(
    x,
    tip_arr.min(axis=0),
    tip_arr.max(axis=0),
    color="C3",
    alpha=0.5,
    linewidth=0,
    label="Tipping trajectories",
)
ax[1].plot(x, tip_arr.mean(axis=0), color="C3", lw=3)

ax[1].fill_between(
    x,
    nontip_arr.min(axis=0),
    nontip_arr.max(axis=0),
    color="C0",
    alpha=0.5,
    linewidth=0,
    label="Non-tipping trajectories",
)
ax[1].plot(x, nontip_arr.mean(axis=0), color="C0", lw=3)

ax[1].axvline(cutoff_idx, color="0.4", linestyle="--", lw=2, label="Tipping point")
ax[1].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[1].set_ylabel("AMOC Strength [Sv]", fontsize=LABEL_FONTSIZE)
ax[1].grid(alpha=0.2)
ax[1].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

# 3) Parameter H(t)
ax[2].plot(tipping_full_params[0], color="C3", lw=3, label=r"Tipping $H(t)$")
ax[2].plot(non_tipping_full_params[0], color="C0", lw=3, label=r"Non-tipping $H(t)$")
ax[2].axhline(info["system_config"]["H_fold"], color="0.4", linestyle="--", lw=2, label="Threshold")
ax[2].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[2].set_ylabel("Parameter", fontsize=LABEL_FONTSIZE)
ax[2].grid(alpha=0.2)
ax[2].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

for a in ax:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
    a.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/b_amoc3b_traj.png", dpi=300, bbox_inches="tight")
plt.show()


## Compute Early Warning Signals

Apply EWS indicators (AR1, Variance) to each trajectory and measure trend with Kendall τ.

In [ ]:
# EWS parameters
wL = 0.5  # Rolling window length (fraction of time series)
N_ENSEMBLES = 1 # Number of ensembles for the ResKMD runs



def compute_ews_metrics(trajectories, wL, controls=None):
    """Compute EWS metrics for a list of trajectories.

    If controls is provided, it must be a list of 1D arrays (one per trajectory),
    each of length cutoff_idx. ResDMD with control is then computed and returned
    as 'resdmd_control_series'.
    """
    ar1_series = []
    var_series = []
    resdmd_series = []
    resdmd_mlp_series = []
    spectral_gap_list = []
    resdmd_control_series = [] if controls is not None else None

    for i, traj in enumerate(trajectories):
        data = np.asarray(traj)

        # AR1, variance, ResDMD
        ar1 = rolling_acf(data, window=wL)
        var = rolling_variance(data, window=wL)
        spectral_gap = spectral_gap_series(data, wL)

        ## Ensembling for ResKMD runs
        resdmd_ens = []
        resdmd_mlp_ens = []
        for seed in range(N_ENSEMBLES):
            resdmd = kews.resdmd(data, window=wL, observable=rff(n_features=100, scale=0.2, seed=seed))
            resdmd_mlp = kews.resdmd(data, 
                window=wL,
                observable=mlp(hidden_layers=[8, 4], bottleneck_dim=2, seed=seed),
                fit_kwargs=dict(
                    lr=1e-3,
                    epochs=1000,
                    batch_size=32,
                    shuffle=True
                )
            )
            resdmd_ens.append(np.array(resdmd))
            resdmd_mlp_ens.append(np.array(resdmd_mlp))

        ar1_series.append(np.array(ar1))
        var_series.append(np.array(var))
        resdmd_series.append(np.array(resdmd_ens).mean(axis=0))
        resdmd_mlp_series.append(np.array(resdmd_mlp_ens).mean(axis=0))
        spectral_gap_list.append(spectral_gap)

    out = {
        'ar1_series': ar1_series,
        'var_series': var_series,
        'resdmd_series': resdmd_series,
        'resdmd_mlp_series': resdmd_mlp_series,
        'spectral_gap_series': spectral_gap_list,
    }
    if resdmd_control_series is not None:
        out['resdmd_control_series'] = resdmd_control_series
    return out

# Compute EWS (trajectories are already window [start_ews:cutoff_idx] of length num_timesteps)
tipping_ews = compute_ews_metrics(tipping_trajectories, wL)
non_tipping_ews = compute_ews_metrics(non_tipping_trajectories, wL)


## Visualize EWS Indicators

Compare AR1 and Variance trends between tipping and non-tipping trajectories.

In [ ]:
# Plot EWS
start_idx = 0
t = burnin_idx + start_ews + np.arange(len(tipping_ews['ar1_series'][0]))

def masked_array(series_list, start_idx):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    return arr

def plot_distribution(ax, series_list, color, label, start_idx, band_alpha=0.15):
    arr = masked_array(series_list, start_idx)
    median = np.nanmedian(arr, axis=0)
    q25, q75 = np.nanpercentile(arr, [25, 75], axis=0)
    ax.plot(t, median, color=color, label=label, linewidth=3)
    ax.fill_between(t, q25, q75, color=color, alpha=band_alpha, linewidth=0)

def robust_ylim(series_lists, start_idx, low=2, high=98, margin=0.05):
    """Compute ylim from percentile of valid values so end spikes don't dominate."""
    vals = np.concatenate([masked_array(s, start_idx)[:, start_idx:].ravel() for s in series_lists])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad

panel_specs = [
    ('spectral_gap_series', 'Spectral gap', False),
    ('ar1_series', 'AR1', False),
    ('var_series', 'Variance', False),
    ('resdmd_series', 'ResKMD (RFF)', True),
    ('resdmd_mlp_series', 'ResKMD (MLP)', True),
]

fig, axes = plt.subplots(1, len(panel_specs), figsize=(24, 4), sharex=True)

for i, (key, ylabel, use_robust_ylim) in enumerate(panel_specs):
    ax = axes[i]

    plot_distribution(ax, tipping_ews[key], "C3", "Tipping", start_idx)
    plot_distribution(ax, non_tipping_ews[key], "C0", "Non-tipping", start_idx)

    ax.set_xlabel("Time", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_xlim([t[0], t[-1]])
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

    if use_robust_ylim:
        ax.set_ylim(robust_ylim([tipping_ews[key], non_tipping_ews[key]], start_idx))

    if i == 0:
        ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_amoc3b_ews.png', dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# AUROC benchmark across trajectories
import pandas as pd
from experiments.metrics import ensemble_roc, plot_auroc, plot_roc

y_true = np.r_[np.ones(len(tipping_ews['ar1_series'])), np.zeros(len(non_tipping_ews['ar1_series']))].astype(int)

result = ensemble_roc(
    y_true,
    {
        'AR1': tipping_ews['ar1_series'] + non_tipping_ews['ar1_series'],
        'Variance': tipping_ews['var_series'] + non_tipping_ews['var_series'],
        'ResKMD (RFF)': tipping_ews['resdmd_series'] + non_tipping_ews['resdmd_series'],
        'ResKMD (MLP)': tipping_ews['resdmd_mlp_series'] + non_tipping_ews['resdmd_mlp_series'],
    },
)

auroc_df = pd.DataFrame(
    [
        {
            'indicator': name,
            'auroc': curve.auroc,
            'n_valid': curve.n_valid,
            'n_missing': curve.n_missing,
        }
        for name, curve in result.curves.items()
    ]
).sort_values('auroc', ascending=False).reset_index(drop=True)
display(auroc_df)

fig, ax = plt.subplots(figsize=(6, 4))
plot_auroc(result, ax=ax, label_fontsize=LABEL_FONTSIZE, tick_labelsize=TICK_FONTSIZE)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_amoc3b_auroc.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(
    result,
    ax=ax,
    title=None,
    chance=True,
    show_auroc=True,
    linewidth=3,
    bold_top_n=2,
    label_fontsize=LABEL_FONTSIZE,
    tick_labelsize=TICK_FONTSIZE,
    legend_fontsize=LEGEND_FONTSIZE,
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/b_amoc3b_roc.png', dpi=300, bbox_inches='tight')
plt.show()
